# Compose As You Are — Python Music Lab

## Google Colabスターター

ブラウザアプリで学んだ変数、リスト、`for`、`if`、乱数、関数を使い、Colab上で自分の曲を発展させます。

このノートブックでは、次の順で進みます。

1. アプリと同じ教育用作曲APIを準備する
2. Pythonコードで曲データを作る
3. 曲データを表とピアノロールで確認する
4. NumPyで簡単な音声波形を合成する
5. 独自の作曲規則やデータ可聴化へ発展させる

最初は上から順番に実行し、**「自分の曲を書く」セルだけを少しずつ変更**してください。

## 1. 作曲APIを準備する

このノートブックの中に、ブラウザアプリと同じ教育用作曲APIが収録されています。次のセルを実行すると、以降のセルから`caya_music`を読み込めます。


In [ ]:
from pathlib import Path

# ブラウザ版と同じ教育用作曲APIを、ノートブック内から準備します。
CAYA_MUSIC_SOURCE = r'''"""Compose As You Are の教育用作曲API。

このモジュールは、Pythonコードから音符イベントを作るための小さなAPIです。
音そのものはブラウザ側で再生します。ここでは曲を辞書データとして組み立てます。
"""

from __future__ import annotations

from copy import deepcopy
import math
import re
from typing import Sequence

__all__ = [
    "start_song",
    "set_tempo",
    "set_loop",
    "new_track",
    "use_track",
    "move_to",
    "current_beat",
    "add_note",
    "add_chord",
    "add_rest",
    "add_drum",
    "export_song",
]

_SCHEMA_VERSION = "1.0"
_DEFAULT_TEMPO = 96
_DEFAULT_INSTRUMENT = "soft_synth"
_DEFAULT_TRACK_NAME = "melody"
_MAX_TRACKS = 12
_MAX_EVENTS = 2048
_MAX_BEATS = 512.0
_MAX_DURATION = 64.0

_ALLOWED_INSTRUMENTS = {
    "soft_synth",
    "pluck",
    "bell",
    "warm_pad",
    "bass",
    "drums",
}
_ALLOWED_DRUMS = {"kick", "snare", "hihat", "clap"}
_NOTE_PATTERN = re.compile(r"^[A-Ga-g](?:#|b)?-?\d{1,2}$")

_song: dict
_current_track: str | None
_event_count: int


def _new_song() -> dict:
    return {
        "schema_version": _SCHEMA_VERSION,
        "title": "Untitled",
        "tempo": _DEFAULT_TEMPO,
        "time_signature": [4, 4],
        "loop": False,
        "tracks": [],
        "length_beats": 0.0,
    }


def _reset() -> None:
    global _song, _current_track, _event_count
    _song = _new_song()
    _current_track = None
    _event_count = 0


_reset()


def _number(value: object, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{name} は数値で指定してください。")
    value = float(value)
    if not math.isfinite(value):
        raise ValueError(f"{name} には有限の数値を指定してください。")
    return value


def _bounded(value: object, name: str, lower: float, upper: float) -> float:
    number = _number(value, name)
    if not lower <= number <= upper:
        raise ValueError(f"{name} は {lower:g} 以上 {upper:g} 以下にしてください。")
    return number


def _normalise_instrument(instrument: object) -> str:
    if not isinstance(instrument, str):
        raise TypeError("instrument は文字列で指定してください。")
    instrument = instrument.strip()
    if instrument not in _ALLOWED_INSTRUMENTS:
        choices = ", ".join(sorted(_ALLOWED_INSTRUMENTS))
        raise ValueError(f"instrument は次のいずれかにしてください: {choices}")
    return instrument


def _normalise_note(note: object) -> str:
    if not isinstance(note, str):
        raise TypeError("note は 'C4' のような文字列で指定してください。")
    note = note.strip()
    if not _NOTE_PATTERN.fullmatch(note):
        raise ValueError(
            f"'{note}' は音名として読み取れません。'C4'、'F#4'、'Bb3' のように書いてください。"
        )
    letter = note[0].upper()
    normalised = letter + note[1:]

    pitch_names = {
        "C": 0, "C#": 1, "Db": 1, "D": 2, "D#": 3, "Eb": 3,
        "E": 4, "F": 5, "F#": 6, "Gb": 6, "G": 7, "G#": 8,
        "Ab": 8, "A": 9, "A#": 10, "Bb": 10, "B": 11,
    }
    match = re.fullmatch(r"([A-G](?:#|b)?)(-?\d{1,2})", normalised)
    assert match is not None
    pitch, octave_text = match.groups()
    midi = (int(octave_text) + 1) * 12 + pitch_names[pitch]
    if not 0 <= midi <= 127:
        raise ValueError(
            f"'{normalised}' は再生できる音域の外です。C-1からG9の範囲を目安にしてください。"
        )
    return normalised


def _normalise_notes(notes: object) -> list[str]:
    if isinstance(notes, str):
        return [_normalise_note(notes)]
    if not isinstance(notes, Sequence):
        raise TypeError("notes は音名のリストで指定してください。")
    result = [_normalise_note(note) for note in notes]
    if not result:
        raise ValueError("notes のリストを空にすることはできません。")
    if len(result) > 12:
        raise ValueError("一度に鳴らす音は12音以下にしてください。")
    return result


def _track_by_name(name: str) -> dict:
    for track in _song["tracks"]:
        if track["name"] == name:
            return track
    raise ValueError(f"'{name}' というトラックはありません。")


def _ensure_track() -> dict:
    global _current_track
    if _current_track is None:
        new_track(_DEFAULT_TRACK_NAME, instrument=_DEFAULT_INSTRUMENT)
    return _track_by_name(_current_track)  # type: ignore[arg-type]


def _register_event(track: dict, event: dict, duration: float) -> None:
    global _event_count
    if _event_count >= _MAX_EVENTS:
        raise RuntimeError(f"1回に作れるイベントは {_MAX_EVENTS} 個までです。")
    end = event["beat"] + duration
    if end > _MAX_BEATS:
        raise ValueError(f"曲の長さは {_MAX_BEATS:g} 拍以内にしてください。")
    track["events"].append(event)
    track["cursor"] = max(track["cursor"], end)
    _song["length_beats"] = max(_song["length_beats"], end)
    _event_count += 1


def start_song(
    tempo: int | float = _DEFAULT_TEMPO,
    instrument: str = _DEFAULT_INSTRUMENT,
    title: str = "Untitled",
    volume: int | float = 0.8,
    pan: int | float = 0.0,
) -> None:
    """新しい曲を開始し、最初のトラックを作ります。

    volumeとpanは省略できます。作曲スタジオでは最初の旋律トラックの
    音量と左右位置を設定するために使います。
    """
    _reset()
    set_tempo(tempo)
    if not isinstance(title, str):
        raise TypeError("title は文字列で指定してください。")
    _song["title"] = title.strip() or "Untitled"
    new_track(_DEFAULT_TRACK_NAME, instrument=instrument, volume=volume, pan=pan)


def set_tempo(tempo: int | float) -> None:
    """テンポをBPMで設定します。"""
    value = _bounded(tempo, "tempo", 40, 240)
    _song["tempo"] = int(value) if value.is_integer() else value


def set_loop(enabled: bool = True) -> None:
    """曲を繰り返し再生するかどうかを曲データに記録します。"""
    if not isinstance(enabled, bool):
        raise TypeError("enabled は True または False で指定してください。")
    _song["loop"] = enabled


def new_track(
    name: str,
    instrument: str = _DEFAULT_INSTRUMENT,
    volume: int | float = 0.8,
    pan: int | float = 0.0,
) -> None:
    """新しいトラックを作り、そのトラックを編集対象にします。"""
    global _current_track
    if not isinstance(name, str):
        raise TypeError("name は文字列で指定してください。")
    name = name.strip()
    if not name:
        raise ValueError("トラック名を空にすることはできません。")
    if len(name) > 40:
        raise ValueError("トラック名は40文字以内にしてください。")
    if any(track["name"] == name for track in _song["tracks"]):
        raise ValueError(f"'{name}' というトラックは既にあります。")
    if len(_song["tracks"]) >= _MAX_TRACKS:
        raise RuntimeError(f"トラックは {_MAX_TRACKS} 個まで作れます。")

    track = {
        "name": name,
        "instrument": _normalise_instrument(instrument),
        "volume": _bounded(volume, "volume", 0.0, 1.0),
        "pan": _bounded(pan, "pan", -1.0, 1.0),
        "events": [],
        "cursor": 0.0,
    }
    _song["tracks"].append(track)
    _current_track = name


def use_track(name: str) -> None:
    """既に作ったトラックを編集対象にします。"""
    global _current_track
    if not isinstance(name, str):
        raise TypeError("name は文字列で指定してください。")
    _track_by_name(name)
    _current_track = name


def move_to(beat: int | float) -> None:
    """現在のトラックで、次の音を置く開始位置を変更します。"""
    track = _ensure_track()
    value = _bounded(beat, "beat", 0.0, _MAX_BEATS)
    track["cursor"] = value


def current_beat() -> float:
    """現在のトラックで、次の音を置く開始位置を返します。"""
    return float(_ensure_track()["cursor"])


def add_note(
    note: str | Sequence[str],
    duration: int | float = 1.0,
    velocity: int | float = 0.8,
    beat: int | float | None = None,
) -> None:
    """1音、または複数音の和音を追加します。"""
    notes = _normalise_notes(note)
    duration_value = _bounded(duration, "duration", 0.03125, _MAX_DURATION)
    velocity_value = _bounded(velocity, "velocity", 0.0, 1.0)
    track = _ensure_track()
    start = track["cursor"] if beat is None else _bounded(beat, "beat", 0.0, _MAX_BEATS)
    event = {
        "kind": "note",
        "beat": float(start),
        "duration": duration_value,
        "notes": notes,
        "velocity": velocity_value,
    }
    _register_event(track, event, duration_value)


def add_chord(
    notes: Sequence[str],
    duration: int | float = 1.0,
    velocity: int | float = 0.75,
    beat: int | float | None = None,
) -> None:
    """音名のリストを和音として追加します。"""
    add_note(notes, duration=duration, velocity=velocity, beat=beat)


def add_rest(duration: int | float = 1.0) -> None:
    """現在のトラックの時刻を進め、休符を作ります。"""
    track = _ensure_track()
    duration_value = _bounded(duration, "duration", 0.03125, _MAX_DURATION)
    end = track["cursor"] + duration_value
    if end > _MAX_BEATS:
        raise ValueError(f"曲の長さは {_MAX_BEATS:g} 拍以内にしてください。")
    track["cursor"] = end
    _song["length_beats"] = max(_song["length_beats"], end)


def add_drum(
    sound: str,
    duration: int | float = 0.25,
    velocity: int | float = 0.8,
    beat: int | float | None = None,
) -> None:
    """キック、スネア、ハイハット、クラップの打楽器音を追加します。"""
    if not isinstance(sound, str):
        raise TypeError("sound は文字列で指定してください。")
    sound = sound.strip().lower()
    if sound not in _ALLOWED_DRUMS:
        choices = ", ".join(sorted(_ALLOWED_DRUMS))
        raise ValueError(f"sound は次のいずれかにしてください: {choices}")
    duration_value = _bounded(duration, "duration", 0.03125, 4.0)
    velocity_value = _bounded(velocity, "velocity", 0.0, 1.0)
    track = _ensure_track()
    start = track["cursor"] if beat is None else _bounded(beat, "beat", 0.0, _MAX_BEATS)
    event = {
        "kind": "drum",
        "beat": float(start),
        "duration": duration_value,
        "sound": sound,
        "velocity": velocity_value,
    }
    _register_event(track, event, duration_value)


def export_song() -> dict:
    """ブラウザへ渡すための曲データを返します。"""
    result = deepcopy(_song)
    for track in result["tracks"]:
        track.pop("cursor", None)
        track["events"].sort(key=lambda event: (event["beat"], event["kind"]))
    result["length_beats"] = float(result["length_beats"])
    return result
'''

api_path = Path("caya_music.py")
api_path.write_text(CAYA_MUSIC_SOURCE, encoding="utf-8")
print("教育用作曲APIを準備しました:", api_path.resolve())


## 2. 自分の曲を書く

次のコードはブラウザアプリと同じ形です。まずそのまま実行し、次に`tempo`、`seed`、`notes`、条件式などを1か所ずつ変更してください。

In [ ]:
from caya_music import *
import random

# 変更しやすい設定
tempo = 108
sound = "pluck"
seed = 7
notes = ["C4", "D4", "E4", "G4", "A4", "C5"]

random.seed(seed)

start_song(
    tempo=tempo,
    instrument=sound,
    title="Colabで作る自動作曲",
)

for i in range(24):
    # 4回ごとの最後を休符にする
    if i % 4 == 3:
        add_rest(0.5)
    else:
        note = random.choice(notes)
        velocity = random.choice([0.55, 0.70, 0.85])
        add_note(note, 0.5, velocity)

song = export_song()
print("曲名:", song["title"])
print("テンポ:", song["tempo"], "BPM")
print("長さ:", song["length_beats"], "拍")

## 3. 曲データを表で読む

Pythonコードが直接音を鳴らしているのではなく、まず音符の開始位置、長さ、音名、強さを持つ辞書データを作っています。

In [ ]:
import pandas as pd

rows = []
for track in song["tracks"]:
    for event in track["events"]:
        if event["kind"] == "note":
            label = " + ".join(event["notes"])
        else:
            label = event["sound"]
        rows.append({
            "track": track["name"],
            "instrument": track["instrument"],
            "beat": event["beat"],
            "duration": event["duration"],
            "sound": label,
            "velocity": event["velocity"],
        })

song_table = pd.DataFrame(rows)
song_table

## 4. ピアノロールを描く

横軸は時間、縦軸は音の高さです。コードの反復や条件分岐が、図の規則として見えるか確認してください。

In [ ]:
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

NOTE_PATTERN = re.compile(r"^([A-Ga-g])([#b]?)(-?\d+)$")
SEMITONES = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}


def note_to_midi(note):
    match = NOTE_PATTERN.fullmatch(note)
    if match is None:
        raise ValueError(f"音名を読み取れません: {note}")
    letter, accidental, octave_text = match.groups()
    pitch = SEMITONES[letter.upper()]
    if accidental == "#":
        pitch += 1
    elif accidental == "b":
        pitch -= 1
    octave = int(octave_text)
    return 12 * (octave + 1) + pitch


fig, ax = plt.subplots(figsize=(12, 5))
track_count = max(1, len(song["tracks"]))

for track_index, track in enumerate(song["tracks"]):
    for event in track["events"]:
        if event["kind"] != "note":
            continue
        for note in event["notes"]:
            midi = note_to_midi(note)
            rectangle = Rectangle(
                (event["beat"], midi - 0.4),
                event["duration"],
                0.8,
                alpha=0.55 + 0.35 * event["velocity"],
            )
            ax.add_patch(rectangle)

ax.set_xlim(0, max(1, song["length_beats"]))
all_midis = [
    note_to_midi(note)
    for track in song["tracks"]
    for event in track["events"]
    if event["kind"] == "note"
    for note in event["notes"]
]
if all_midis:
    ax.set_ylim(min(all_midis) - 2, max(all_midis) + 2)
ax.set_xlabel("Beat")
ax.set_ylabel("Pitch (MIDI note number)")
ax.set_title(song["title"])
ax.grid(True, alpha=0.25)
plt.show()

## 5. 曲データから音声を合成する

次のセルは、音符を単純な波形へ変換する準備です。最初は内部の数式をすべて理解する必要はありません。興味を持った場合は、周波数、包絡線、倍音、サンプリング周波数を調べてください。

In [ ]:
import math
import numpy as np
from IPython.display import Audio, display


def note_to_frequency(note):
    midi = note_to_midi(note)
    return 440.0 * 2 ** ((midi - 69) / 12)


def make_envelope(length, sample_rate, attack=0.012, release=0.10):
    envelope = np.ones(length)
    attack_samples = min(length, max(1, int(attack * sample_rate)))
    release_samples = min(length, max(1, int(release * sample_rate)))
    envelope[:attack_samples] = np.linspace(0, 1, attack_samples)
    envelope[-release_samples:] *= np.linspace(1, 0, release_samples)
    return envelope


def synthesise_song(song, sample_rate=22050):
    seconds_per_beat = 60.0 / song["tempo"]
    total_seconds = song["length_beats"] * seconds_per_beat + 0.3
    audio = np.zeros(int(total_seconds * sample_rate), dtype=np.float64)
    random_generator = np.random.default_rng(0)

    for track in song["tracks"]:
        track_gain = float(track.get("volume", 0.8))
        for event in track["events"]:
            start = int(event["beat"] * seconds_per_beat * sample_rate)
            duration_seconds = event["duration"] * seconds_per_beat
            length = max(1, int(duration_seconds * sample_rate))
            end = min(len(audio), start + length)
            length = end - start
            if length <= 0:
                continue

            t = np.arange(length) / sample_rate
            velocity = float(event["velocity"])
            envelope = make_envelope(length, sample_rate)

            if event["kind"] == "note":
                wave = np.zeros(length)
                for note in event["notes"]:
                    frequency = note_to_frequency(note)
                    fundamental = np.sin(2 * math.pi * frequency * t)
                    second_harmonic = 0.22 * np.sin(2 * math.pi * 2 * frequency * t)
                    wave += fundamental + second_harmonic
                wave /= max(1, len(event["notes"]))
            else:
                sound = event.get("sound", "hihat")
                noise = random_generator.normal(0, 1, length)
                if sound == "kick":
                    frequency = np.linspace(130, 48, length)
                    phase = 2 * math.pi * np.cumsum(frequency) / sample_rate
                    wave = np.sin(phase) * np.exp(-8 * t)
                elif sound == "snare":
                    wave = 0.75 * noise * np.exp(-13 * t)
                elif sound == "clap":
                    wave = noise * np.exp(-18 * t)
                else:
                    wave = 0.45 * noise * np.exp(-35 * t)

            audio[start:end] += track_gain * velocity * envelope * wave

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = 0.90 * audio / peak
    return audio, sample_rate


audio_data, sample_rate = synthesise_song(song)
display(Audio(audio_data, rate=sample_rate))

## 6. 小さな発展課題

次の課題から1つを選び、変更前後のコードと結果を記録してください。

- `seed`だけを3通り変え、旋律の違いを比較する
- `notes`へ同じ音を複数回入れ、選ばれる確率を変える
- `i % 4 == 3`を別の条件へ変え、休符の周期を変える
- 音の長さをリストからランダムに選ぶ
- `make_phrase(notes, repeats)`という関数を作る
- メロディにベースまたはドラムのトラックを加える

In [ ]:
# ここへ自分の発展コードを書きます。
# 最初は1か所だけ変更し、song = export_song()を最後に実行してください。

## 7. データを音へ変換する例

数値データを、音の高さの番号へ変換します。下の温度は模擬データです。自分の学科や関心に合うデータへ置き換えてください。

In [ ]:
from caya_music import *

values = [18, 19, 21, 24, 27, 29, 28, 25, 22, 20]
scale = ["C4", "D4", "E4", "G4", "A4", "C5"]

start_song(
    tempo=92,
    instrument="warm_pad",
    title="数値データの可聴化",
)

minimum = min(values)
maximum = max(values)
width = maximum - minimum

for value in values:
    ratio = (value - minimum) / width
    index = round(ratio * (len(scale) - 1))
    note = scale[index]
    add_note(note, 0.75, 0.68)
    print(value, "→", note)

data_song = export_song()
data_audio, data_rate = synthesise_song(data_song)
display(Audio(data_audio, rate=data_rate))

## 振り返り

次の文章を完成させてください。

- 私が使ったPython文法は、＿＿＿＿＿＿＿＿である。
- ＿＿行目の＿＿＿＿を変更すると、音は＿＿＿＿＿＿＿＿と変化した。
- この自動作曲で人が決めた規則は、＿＿＿＿＿＿＿＿である。
- 次に試したい改善は、＿＿＿＿＿＿＿＿である。